# 셔틀콕 전용 YOLO 모델 학습
- 클래스: shuttlecock 1개만
- 데이터: shuttle_dataset.zip (1,591장)
- 10 에폭마다 Drive 자동 저장

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil, os

# 데이터셋 압축 해제
shutil.copy('/content/drive/MyDrive/shuttle_dataset.zip', '/content/shuttle_dataset.zip')
!unzip -q /content/shuttle_dataset.zip -d /content/
!ls /content/shuttle_dataset/
!cat /content/shuttle_dataset/data.yaml

In [ ]:
!pip install ultralytics -q

In [ ]:
import shutil, os
from ultralytics import YOLO
from ultralytics.utils.callbacks.base import default_callbacks

DRIVE_SAVE_DIR = '/content/drive/MyDrive/shuttle_only_checkpoints'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

# ── 자동 저장 콜백 (10 에폭마다 + 매 best 갱신 시) ──
def on_train_epoch_end(trainer):
    epoch = trainer.epoch + 1
    # 10 에폭마다 저장
    if epoch % 10 == 0:
        src = str(trainer.last)  # last.pt 경로
        dst = f'{DRIVE_SAVE_DIR}/epoch{epoch:03d}.pt'
        shutil.copy(src, dst)
        print(f'\n[자동저장] epoch{epoch:03d}.pt → Drive 저장 완료')

def on_train_end(trainer):
    # 학습 완료 시 best.pt 저장
    shutil.copy(str(trainer.best), f'{DRIVE_SAVE_DIR}/best_shuttle_only.pt')
    shutil.copy(str(trainer.best), '/content/drive/MyDrive/best_shuttle_only.pt')
    print('\n[완료] best_shuttle_only.pt → Drive 저장 완료')

model = YOLO('yolov8n.pt')
model.add_callback('on_train_epoch_end', on_train_epoch_end)
model.add_callback('on_train_end', on_train_end)

results = model.train(
    data='/content/shuttle_dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    name='shuttle_only_v1',
    project='/content/runs',
    patience=20,
    save=True,
    save_period=10,   # 10 에폭마다 로컬 체크포인트도 저장
    plots=True,
)

In [ ]:
# 검증
model_best = YOLO('/content/drive/MyDrive/best_shuttle_only.pt')
metrics = model_best.val(data='/content/shuttle_dataset/data.yaml')
print(f'mAP50:    {metrics.box.map50:.3f}')
print(f'mAP50-95: {metrics.box.map:.3f}')
print(f'shuttlecock mAP50: {metrics.box.maps[0]:.3f}')